# 第 5 章 · 因果自注意力与 GQA / KV Cache

> 📄 配套动画网页：`../05_attention.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第二部分 · 模型

注意力是 Transformer 的心脏。这一章要写的是全项目最核心的一段代码。

---


## 🎯 本章目标

- 理解缩放点积注意力的四步流程
- 掌握 `matmul` 批量矩阵乘、`masked_fill`、`softmax`
- 理解因果掩码为什么用上三角
- 理解 GQA 的 `repeat_kv` 广播技巧，以及 KV Cache 的作用

---


## 🖼 官方配图

![注意力在整体架构中的位置：每个 Block 的核心就是它（8 个 Q 头 / 4 个 KV 头）](../images/LLM-structure.jpg)

*注意力在整体架构中的位置：每个 Block 的核心就是它（8 个 Q 头 / 4 个 KV 头）*

---


## 📖 背景：四步走完注意力

```python
scores = (Q @ K.T) / sqrt(head_dim)   # 1. 打分
scores = scores + causal_mask          # 2. 遮住未来
attn   = softmax(scores, dim=-1)       # 3. 归一化
out    = attn @ V                      # 4. 加权求和
```

**因果**：位置 t 只能看到 ≤ t 的信息，所以要把注意力矩阵的**上三角**设成 −inf。
**GQA**：8 个 Q 头共享 4 组 KV，KV 显存减半。
**KV Cache**：生成第 t 个词时复用前 t−1 个词的 K/V，每步只算 1 个新 token。

---



# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 5.1 批量矩阵乘：`@` / `torch.matmul`

注意力里所有的乘法都是批量矩阵乘，形状要对齐。

In [ ]:
import torch

Q = torch.randn(2, 8, 6, 16)    # [B, H, S, D]
K = torch.randn(2, 8, 6, 16)

# 只要最后两维做矩阵乘，前面维度自动广播
scores = Q @ K.transpose(-2, -1)
print("scores 形状:", scores.shape, " <- [B, H, S, S]")

### 5.2 因果掩码：`masked_fill` + `triu`

`triu(diagonal=1)` 取上三角（不含对角线），正是「不能看未来」要遮住的部分。

In [ ]:
S = 5
mask = torch.triu(torch.ones(S, S), diagonal=1).bool()
print("上三角布尔掩码:")
print(mask.int())

scores = torch.randn(1, 1, S, S)
scores = scores.masked_fill(mask, float("-inf"))
print("\n遮罩后的最后一行:", scores[0, 0, -1].tolist(), " <- 只有最后一个是有限值")

### 5.3 `softmax` 与数值稳定性

`−inf` 经过 `exp` 会变成 0，正好实现「权重为 0」。

In [ ]:
x = torch.tensor([[1.0, 2.0, float("-inf"), 3.0]])
print("softmax:", torch.softmax(x, dim=-1).tolist(), " <- -inf 位置变成 0")

### 5.4 `repeat_kv`：GQA 的关键技巧

`[B, S, kv_heads, D]` 要变成 `[B, S, q_heads, D]`，让每 `n_rep` 个 Q 头共享一个 KV 头。
用 `expand` + `reshape` 完成，**不复制数据**。

In [ ]:
x = torch.randn(1, 6, 2, 4)          # kv_heads = 2
n_rep = 2                            # 每个 kv 头被 2 个 q 头共享

bs, slen, kvh, d = x.shape
y = x[:, :, :, None, :].expand(bs, slen, kvh, n_rep, d).reshape(bs, slen, kvh * n_rep, d)
print("原始:", x.shape, " -> 重复后:", y.shape)
print("验证前两组是否相同:", torch.equal(y[:, :, 0], y[:, :, 1]))

### 5.5 内置加速：`F.scaled_dot_product_attention`

PyTorch 2.x 提供了融合实现的 SDPA，内部会自动选 FlashAttention / 内存高效核。

In [ ]:
import torch.nn.functional as F

q = torch.randn(1, 4, 6, 16)
k = torch.randn(1, 4, 6, 16)
v = torch.randn(1, 4, 6, 16)

out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("SDPA 输出形状:", out.shape)

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

本章要写的是全项目最核心的一段代码。请务必自己实现一遍。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 5.6 实现 `repeat_kv`

先写这个小工具——GQA 要靠它把 KV 头「复制」到和 Q 头一样多。

输入 `[B, S, kv_heads, D]` 和 `n_rep`，输出 `[B, S, kv_heads*n_rep, D]`。

In [ ]:
# ======== 你的实现：repeat_kv ========
#
# 【要求】
#   - 如果 n_rep == 1，直接返回原张量
#   - 否则用 x[:, :, :, None, :].expand(...) 再 reshape
#   - 保证结果中每 n_rep 个相邻头的数据相同
#
# 【可用到的函数 / 方法】
#   - `Tensor.expand`、`Tensor.reshape`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：repeat_kv ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    x = torch.randn(2, 6, 2, 4)
    y = repeat_kv(x, 2)
    assert tuple(y.shape) == (2, 6, 4, 4), f"形状应为 (2, 6, 4, 4)，得到 {tuple(y.shape)}"
    same(y[:, :, 0], y[:, :, 1], msg="第 0、1 个头应完全相同")
    same(y[:, :, 2], y[:, :, 3], msg="第 2、3 个头应完全相同")
    assert tuple(repeat_kv(x, 1).shape) == tuple(x.shape), "n_rep=1 时形状不应改变"

check("repeat_kv 形状与分组", _t)

### 5.7 实现 `causal_attention`

**要求**：实现一个函数，输入已经投影好的 Q / K / V，返回注意力输出。
**注意张量布局和 `repeat_kv` 保持一致**（头在第 2 维，不在第 1 维）：

```python
def causal_attention(q, k, v, n_rep=1, is_causal=True):
    # q:  [B, S, H,   D]
    # k:  [B, S, Hkv, D]      H == Hkv * n_rep
    # v:  [B, S, Hkv, D]
    # 返回 [B, S, H, D]
```

步骤：
1. 先用 `repeat_kv` 把 k/v 的头数扩到 H
2. `transpose(1, 2)` 变成 `[B, H, S, D]`，再算 `scores = q @ kᵀ / sqrt(D)`
3. 因果掩码 → `softmax` → `@ v`
4. 再 `transpose(1, 2)` 变回 `[B, S, H, D]`

In [ ]:
# ======== 你的实现：causal_attention ========
#
# 【要求】
#   - 签名：causal_attention(q, k, v, n_rep=1, is_causal=True)
#   - 输入 q: [B, S, H, D]，k/v: [B, S, Hkv, D]；输出 [B, S, H, D]
#   - 先用 repeat_kv 把 k/v 的头数从 Hkv 扩到 H
#   - transpose(1, 2) 得到 [B, H, S, D]，scores = q @ k.transpose(-2, -1) / sqrt(D)
#   - is_causal=True 时把上三角（对角线以上）设成 -inf
#   - softmax 后与 v 相乘，再 transpose(1, 2) 还原布局
#
# 【可用到的函数 / 方法】
#   - `Tensor.transpose`、`@` 或 `torch.matmul`
#   - `torch.triu`、`Tensor.masked_fill`
#   - `torch.softmax`、`math.sqrt`
#   - 上面写过的 `repeat_kv`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：causal_attention ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    B, S, H, D = 1, 5, 2, 4
    q = torch.randn(B, S, H, D); k = torch.randn(B, S, H, D); v = torch.randn(B, S, H, D)
    out = causal_attention(q, k, v)
    assert tuple(out.shape) == (B, S, H, D), f"输出形状应为 {(B, S, H, D)}，得到 {tuple(out.shape)}"
    q2 = q.clone(); q2[:, -1] += 5.0
    close(causal_attention(q2, k, v)[:, :-1], out[:, :-1], tol=1e-5,
          msg="只改了最后一个位置的 q，却影响了前面的输出 —— 因果掩码不对")
    ref = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
                                         is_causal=True).transpose(1, 2)
    close(out, ref, tol=1e-4, msg="与 F.scaled_dot_product_attention(is_causal=True) 不一致")
    q3 = torch.randn(1, 5, 4, 4); k3 = torch.randn(1, 5, 2, 4); v3 = torch.randn(1, 5, 2, 4)
    assert tuple(causal_attention(q3, k3, v3, n_rep=2).shape) == (1, 5, 4, 4), "n_rep=2（GQA）时形状不对"

check("causal_attention 形状 / 因果性 / 与 SDPA 一致", _t)

### 5.8 验证因果性

写一个测试：把序列**最后一个位置**的输入改掉，前面位置的输出应该**完全不变**（因为看不到未来）。

In [ ]:
# ======== 你的实现：verify_causal ========
#
# 【要求】
#   - 构造输入 x: [1, 6, 32]
#   - 跑一遍得到 out1
#   - 把 x[:, -1, :] 随机改掉，再跑一遍得到 out2
#   - 断言 out1[:, :-1, :] 与 out2[:, :-1, :] 完全一致
#
# 【可用到的函数 / 方法】
#   - `torch.allclose`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能说清 scores 的形状是 `[B, H, S, S]`
- [ ] 因果掩码为什么用 `triu(diagonal=1)`？
- [ ] `repeat_kv` 有没有真的复制数据？（expand 是视图）
- [ ] 因果性验证通过了吗？

**思考题**：KV Cache 让推理复杂度从 O(t²) 降到 O(t)，那训练时为什么不用它？
